# Generate routes.txt and shapes.txt

Generates GTFS routes and shapes files from scraped route geometries.

In [1]:
import json
from pathlib import Path 
import pandas as pd
import geopandas as gpd

## Parameters

In [2]:
# Path to params.json (same directory as this notebook)
_params_path = "../params.json"
with open(_params_path, encoding="utf-8") as f:
    p = json.load(f)

In [3]:
CITY = p["city"]
AGENCY_ID = p["agency"]["id"]

In [4]:
# --- GTFS folder ---
PATH_DIR_GTFS = Path(f"../data/{CITY}/gtfs-frequencies")
PATH_DIR_GTFS.mkdir(parents=True, exist_ok=True)
print(f"Output: {PATH_DIR_GTFS.absolute()}")

# --- Processed folder ---
PATH_DIR_processed = Path(f"../data/{CITY}/processed")
PATH_DIR_processed.mkdir(parents=True, exist_ok=True)
print(f"Output: {PATH_DIR_processed.absolute()}")

Output: /Users/danielbustillos/Documents/ITDP/proyectos-ITDP/generacionDatos/generation-gtfs-from-shapes/1-gtfs-generation/../data/merida/gtfs-frequencies
Output: /Users/danielbustillos/Documents/ITDP/proyectos-ITDP/generacionDatos/generation-gtfs-from-shapes/1-gtfs-generation/../data/merida/processed


## Read and process scraped routes

### Read shapes from raw geojson

In [5]:
### Read shapes from raw geojson
path_routes_raw = Path(f"../data/{CITY}/routes-shapes/{CITY}.geojson")
print(f"Raw shapes path: {path_routes_raw.absolute()}")

## Read shapes from raw geojson
routes_raw = gpd.read_file(path_routes_raw)
print(f"Raw shapes (# routes): {routes_raw.shape}")

Raw shapes path: /Users/danielbustillos/Documents/ITDP/proyectos-ITDP/generacionDatos/generation-gtfs-from-shapes/1-gtfs-generation/../data/merida/routes-shapes/merida.geojson
Raw shapes (# routes): (56, 4)


In [6]:
routes_raw.head(2)

,route_name,route_name_short,route_type,geometry
0,2_42 Caseta,42 Caseta,3,"LINESTRING (1478265.739 2346761.139, 1478081.3..."
1,6_42 Sur IMSS,42 Sur IMSS,3,"LINESTRING (1478238.108 2346765.115, 1478079.2..."


GTFS basic format

In [7]:
routes_raw["agency_id"] = AGENCY_ID

In [8]:
routes_raw["route_name"] = routes_raw["route_name"].str.title()
routes_raw["route_name_short"] = routes_raw["route_name_short"].str.title()
routes_raw["shape_id"] = "Shape_" + routes_raw["route_name"]

## 2. Format in GTFS routes format

Format the table and adjust to routes.txt, remove geometry

In [9]:
# A partir de routes_clean y parámetros (AGENCY_ID): tabla GTFS
df_routes_gtfs = pd.DataFrame({
    "route_name": routes_raw["route_name"],
    "agency_id": routes_raw["agency_id"],
    "route_name_short": routes_raw["route_name"],
    "route_long_name": routes_raw["route_name"],
    "route_type": routes_raw["route_type"],
})

df_routes_gtfs.head(5)

,route_name,agency_id,route_name_short,route_long_name,route_type
0,2_42 Caseta,Red_de_Transporte_Merida,2_42 Caseta,2_42 Caseta,3
1,6_42 Sur Imss,Red_de_Transporte_Merida,6_42 Sur Imss,6_42 Sur Imss,3
2,11_50 Penal Paso Texas,Red_de_Transporte_Merida,11_50 Penal Paso Texas,11_50 Penal Paso Texas,3
3,13_50 Sur Villa Magna,Red_de_Transporte_Merida,13_50 Sur Villa Magna,13_50 Sur Villa Magna,3
4,16_Zazil-Ha,Red_de_Transporte_Merida,16_Zazil-Ha,16_Zazil-Ha,3


## Generate shapes.txt

In [10]:
# shapes.txt: shape_id, shape_pt_lat, shape_pt_lon, shape_pt_sequence [, shape_dist_traveled]
# routes_raw has geometry in UTM (32614). Reproject to WGS84 for lat/lon.
from shapely.geometry import Point

routes_wgs84 = routes_raw.to_crs(epsg=4326)

filas = []
for idx in routes_raw.index:
    row_utm = routes_raw.loc[idx]
    row_wgs = routes_wgs84.loc[idx]
    shape_id = row_utm["shape_id"]
    geom_utm = row_utm.geometry
    geom_wgs = row_wgs.geometry
    if geom_wgs is None or geom_wgs.is_empty:
        continue
    coords_wgs = list(geom_wgs.coords)  # (lon, lat)
    # Accumulated distance in km from UTM geometry
    if geom_utm is not None and not geom_utm.is_empty:
        coords_utm = list(geom_utm.coords)
        dist_accum = 0.0
        dists_km = [0.0]
        for i in range(1, len(coords_utm)):
            seg = Point(coords_utm[i - 1]).distance(Point(coords_utm[i]))  # meters
            dist_accum += seg / 1000.0  # km
            dists_km.append(round(dist_accum, 6))
    else:
        dists_km = [0.0] * len(coords_wgs)

    # Omit consecutive duplicate points (same coordinates) to comply with GTFS
    last_pt = None
    seq = 0
    for i, (lon, lat) in enumerate(coords_wgs):
        pt = (round(lat, 6), round(lon, 6))
        if pt == last_pt:
            continue
        last_pt = pt
        seq += 1
        filas.append({
            "shape_id": shape_id,
            "shape_pt_lat": pt[0],
            "shape_pt_lon": pt[1],
            "shape_pt_sequence": seq,
            "shape_dist_traveled": dists_km[i] if i < len(dists_km) else "",
        })

df_shapes = pd.DataFrame(filas)
df_shapes.head()

,shape_id,shape_pt_lat,shape_pt_lon,shape_pt_sequence,shape_dist_traveled
0,Shape_2_42 Caseta,20.96196,-89.62102,1,0.000000
1,Shape_2_42 Caseta,20.96222,-89.62276,2,0.185327
2,Shape_2_42 Caseta,20.96123,-89.62295,3,0.297978
3,Shape_2_42 Caseta,20.96093,-89.62162,4,0.441855
4,Shape_2_42 Caseta,20.95958,-89.61699,5,0.951805


### Shapes validation

Check that for each `shape_id`: (1) `shape_dist_traveled` is always ascending according to `shape_pt_sequence`, and (2) there are no points with the same latitude and longitude (duplicates).

In [11]:
def validate_shapes_gtfs(df):
    """
    Validates that in a DataFrame of shapes (GTFS):
    - shape_dist_traveled is ascending (not decreasing) for each shape_id according to shape_pt_sequence.
    - There are no points with the same (lat, lon) consecutive (duplicates).
    Returns True if everything is valid, False if there are errors. Prints a summary.
    """
    if df.empty or "shape_id" not in df.columns:
        print("Empty DataFrame or without shape_id column.")
        return False

    req = ["shape_id", "shape_pt_lat", "shape_pt_lon", "shape_pt_sequence", "shape_dist_traveled"]
    missing = [c for c in req if c not in df.columns]
    if missing:
        print(f"Missing columns: {missing}")
        return False

    df = df.sort_values(["shape_id", "shape_pt_sequence"]).reset_index(drop=True)
    errores_dist = []
    errores_dup = []

    for shape_id, grp in df.groupby("shape_id", sort=False):
        grp = grp.sort_values("shape_pt_sequence")
        dist = grp["shape_dist_traveled"]
        lat = grp["shape_pt_lat"].round(6)
        lon = grp["shape_pt_lon"].round(6)

        # 1) shape_dist_traveled ascending (each value >= previous)
        if pd.api.types.is_numeric_dtype(dist):
            for i in range(1, len(dist)):
                if dist.iloc[i] < dist.iloc[i - 1]:
                    errores_dist.append((shape_id, int(grp["shape_pt_sequence"].iloc[i]), dist.iloc[i - 1], dist.iloc[i]))
        # If there are empties/string, don't validate dist

        # 2) No (lat, lon) consecutive duplicates
        for i in range(1, len(grp)):
            if (lat.iloc[i] == lat.iloc[i - 1]) and (lon.iloc[i] == lon.iloc[i - 1]):
                errores_dup.append((shape_id, int(grp["shape_pt_sequence"].iloc[i]), lat.iloc[i], lon.iloc[i]))

    ok = len(errores_dist) == 0 and len(errores_dup) == 0
    if errores_dist:
        print(f"[Validation] shape_dist_traveled not ascending: {len(errores_dist)} case(s). Examples: {errores_dist[:5]}")
    if errores_dup:
        print(f"[Validation] (lat, lon) consecutive duplicates: {len(errores_dup)} case(s). Examples: {errores_dup[:5]}")
    if ok:
        print("[Validation] OK: shape_dist_traveled ascending by shape_id and without (lat, lon) consecutive duplicates.")
    return ok


validate_shapes_gtfs(df_shapes)

[Validation] OK: shape_dist_traveled ascending by shape_id and without (lat, lon) consecutive duplicates.


True

## Export files

In [12]:
# Join folder with file name
ruta_final_routes_gtfs = PATH_DIR_GTFS / "routes.txt"
df_routes_gtfs.to_csv(ruta_final_routes_gtfs, index=False) # GTFS routes file

In [13]:
# Join folder with file name
ruta_final_processed_routes = PATH_DIR_processed / "routes_clean.geojson"
# Save directly
routes_raw.to_file(ruta_final_processed_routes, driver="GeoJSON")

In [14]:
# Export to GTFS
path_shapes = PATH_DIR_GTFS / "shapes.txt"
df_shapes.to_csv(path_shapes, index=False, encoding="utf-8")
print(f"Saved: {path_shapes} ({len(df_shapes)} points, {df_shapes['shape_id'].nunique()} shapes)")

Saved: ../data/merida/gtfs-frequencies/shapes.txt (7492 points, 56 shapes)
